In [1]:
# --------------------------------- Part 1: Imports ---------------------------------
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import time
import pickle
import os

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                             roc_curve, precision_recall_curve, auc, accuracy_score)
from scipy.stats import ttest_rel
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Dense, LSTM, Conv1D, Flatten, concatenate, Dropout,
                                     Multiply, Reshape, BatchNormalization, GlobalAveragePooling1D,
                                     Lambda)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
import os
# Save df_final as a .csv file
os.chdir(r'D:\sample_dataset')
df=pd.read_csv('df_final_cleaned.csv', low_memory=False)

In [3]:
df.shape

(1500000, 22)

In [4]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [5]:
# Apply Label Encoding for all object-type columns
label_encoders = {}
for column in df.select_dtypes(include='object').columns:
    le = LabelEncoder()
    df[column] = le.fit_transform(df[column].astype(str))
    label_encoders[column] = le

In [6]:
# Select the features and labels using the column names
X = df.drop(columns=['Label'])
y = df['Label'].values  

In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = scaler.fit_transform(X)

In [8]:
X.shape

(1500000, 21)

In [10]:
y=np.where(y == 0, 0, 1)

In [11]:

# --------------------------------- Part 3: Define Models ---------------------------------
def create_cnn_model(input_shape):
    inputs = Input(shape=input_shape)
    x = Conv1D(64, 3, activation='relu', padding='same')(inputs)
    x = BatchNormalization()(x)
    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    output = Dense(1, activation='sigmoid')(x)
    return Model(inputs, output)

def create_lstm_model(input_shape):
    inputs = Input(shape=input_shape)
    x = LSTM(64, return_sequences=True)(inputs)
    x = BatchNormalization()(x)
    x = LSTM(32)(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    output = Dense(1, activation='sigmoid')(x)
    return Model(inputs, output)

def create_fnn_model(input_shape):
    inputs = Input(shape=(input_shape,))
    x = Dense(128, activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    output = Dense(1, activation='sigmoid')(x)
    return Model(inputs, output)

In [12]:
from memory_profiler import memory_usage

In [13]:
def train_ensemble():
    return ensemble_model.fit(
        [X_train_cnn, X_train_cnn, X_train], y_train,
        epochs=50,
        batch_size=128,
        validation_split=0.1,
        verbose=0,
        validation_data=([X_test_cnn, X_test_cnn, X_test], y_test)
    )

In [33]:
# --------------------------------- Part 4: 5-Fold Cross-Validation Setup ---------------------------------
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

ensemble_accs, cnn_accs, lstm_accs, fnn_accs = [], [], [], []
ensemble_times, cnn_times, lstm_times, fnn_times = [], [], [], []
ensemble_memory_usages, cnn_memory_usages, lstm_memory_usages, fnn_memory_usages = [], [], [], []
all_y_test = []
all_ensemble_pred = []
all_cnn_pred =[]
all_lstm_pred=[]
all_fnn_pred=[]
all_ensemble_prob = []
all_cm = []
fold = 1
for train_idx, test_idx in kfold.split(X, y):
    print(f"\n=== Fold {fold} ===")
    fold += 1

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    X_train_cnn = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
    X_test_cnn = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

    # CNN
   
    cnn_model = create_cnn_model((X_train_cnn.shape[1], 1))
    cnn_model.compile(optimizer=Adam(0.001), loss='binary_crossentropy', metrics=['accuracy'])
    start = time.time()
    cnn_mem_usage, cnn_history = memory_usage(
    (cnn_model.fit, (X_train_cnn, y_train), {'epochs': 50, 'batch_size': 64, 'validation_split': 0.1, 'verbose': 0}),
    interval=0.1,
    retval=True )
    #cnn_history = cnn_model.fit(X_train_cnn, y_train, epochs=10, batch_size=64, validation_split=0.1, verbose=0)
    end = time.time()
    cnn_times.append(end - start)
    cnn_peak_memory = max(cnn_mem_usage)
    cnn_memory_usages.append(cnn_peak_memory)
    cnn_pred = (cnn_model.predict(X_test_cnn,verbose=0) > 0.5).astype(int)
    cnn_accs.append(accuracy_score(y_test, cnn_pred))
    all_cnn_pred.append(cnn_pred)

    # LSTM
   
    lstm_model = create_lstm_model((X_train_cnn.shape[1], 1))
    lstm_model.compile(optimizer=Adam(0.001), loss='binary_crossentropy', metrics=['accuracy'])
    start = time.time()
    lstm_mem_usage, lstm_history = memory_usage(
    (lstm_model.fit, (X_train_cnn, y_train), {'epochs': 50, 'batch_size': 64, 'validation_split': 0.1, 'verbose': 0}),
     interval=0.1,retval=True)
    #lstm_history= lstm_model.fit(X_train_cnn, y_train, epochs=10, batch_size=64, validation_split=0.1, verbose=0)
    end = time.time()
    lstm_times.append(end - start)
    lstm_peak_memory = max(lstm_mem_usage)
    lstm_memory_usages.append(lstm_peak_memory)
    lstm_pred = (lstm_model.predict(X_test_cnn,verbose=0) > 0.5).astype(int)
    lstm_accs.append(accuracy_score(y_test, lstm_pred))
    all_lstm_pred.append(lstm_pred)

    # FNN
    
    fnn_model = create_fnn_model(X_train.shape[1])
    fnn_model.compile(optimizer=Adam(0.001), loss='binary_crossentropy', metrics=['accuracy'])
    start = time.time()
    fnn_mem_usage, fnn_history = memory_usage(
    (fnn_model.fit, (X_train, y_train), {'epochs': 50, 'batch_size': 64, 'validation_split': 0.1, 'verbose': 0}),
    interval=0.1,
    retval=True)
    #fnn_history=fnn_model.fit(X_train, y_train, epochs=10, batch_size=64, validation_split=0.1, verbose=0)
    end = time.time()
    fnn_times.append(end - start)
    fnn_peak_memory = max(fnn_mem_usage)
    fnn_memory_usages.append(fnn_peak_memory)
    fnn_pred = (fnn_model.predict(X_test,verbose=0) > 0.5).astype(int)
    fnn_accs.append(accuracy_score(y_test, fnn_pred))
    all_fnn_pred.append(fnn_pred)

    # Ensemble
   # Static Ensemble
  
    cnn_probs = cnn_model.predict(X_test_cnn,verbose=0)
    lstm_probs = lstm_model.predict(X_test_cnn,verbose=0)
    fnn_probs = fnn_model.predict(X_test,verbose=0)

    # Static average of predicted probabilities
    ensemble_prob = (cnn_probs + lstm_probs + fnn_probs) / 3.0
    ensemble_pred = (ensemble_prob > 0.5).astype(int)

    ensemble_accs.append(accuracy_score(y_test, ensemble_pred))
    ensemble_times.append(cnn_times[-1] + lstm_times[-1] + fnn_times[-1])  # Total time of all models
    ensemble_memory_usages.append(max([cnn_memory_usages[-1], lstm_memory_usages[-1], fnn_memory_usages[-1]]))  # Peak of three

    all_y_test.append(y_test)
    all_ensemble_pred.append(ensemble_pred)
    all_ensemble_prob.append(ensemble_prob)  # For ROC/PR
    cm = confusion_matrix(y_test, ensemble_pred)
    all_cm.append(cm)




=== Fold 1 ===

=== Fold 2 ===

=== Fold 3 ===

=== Fold 4 ===

=== Fold 5 ===


In [49]:

# --------------------------------- Part 5: Report Results ---------------------------------
def report_scores(name, scores, times, memories):
    print(f"{name}: Accuracy = {np.mean(scores):.4f} ± {np.std(scores):.4f}, "
          f"Time = {np.mean(times):.2f}s ± {np.std(times):.2f}s, "
          f"Memory = {np.mean(memories):.2f} MiB ± {np.std(memories):.2f} MiB")

print("\n=== 5-Fold Cross-validation Results ===")
report_scores("CNN", cnn_accs, cnn_times, cnn_memory_usages)
report_scores("LSTM", lstm_accs, lstm_times, lstm_memory_usages)
report_scores("FNN", fnn_accs, fnn_times, fnn_memory_usages)
report_scores("Ensemble", ensemble_accs, ensemble_times, ensemble_memory_usages)


=== 5-Fold Cross-validation Results ===
CNN: Accuracy = 0.8057 ± 0.0007, Time = 2828.55s ± 201.78s, Memory = 713.49 MiB ± 227.49 MiB
LSTM: Accuracy = 0.8345 ± 0.0028, Time = 14283.47s ± 1368.82s, Memory = 831.61 MiB ± 110.70 MiB
FNN: Accuracy = 0.8724 ± 0.0019, Time = 1733.69s ± 178.05s, Memory = 845.10 MiB ± 110.74 MiB
Ensemble: Accuracy = 0.8754 ± 0.0100, Time = 18845.71s ± 1710.39s, Memory = 845.10 MiB ± 110.74 MiB


In [50]:
# --------------------------------- Part 6: Statistical Significance Testing ---------------------------------
print("\n=== Paired t-tests ===")
print("Ensemble vs CNN:", ttest_rel(ensemble_accs, cnn_accs))
print("Ensemble vs LSTM:", ttest_rel(ensemble_accs, lstm_accs))
print("Ensemble vs FNN:", ttest_rel(ensemble_accs, fnn_accs))


=== Paired t-tests ===
Ensemble vs CNN: TtestResult(statistic=13.914975545793027, pvalue=0.00015467300144591083, df=4)
Ensemble vs LSTM: TtestResult(statistic=6.95220401820728, pvalue=0.0022491197557128616, df=4)
Ensemble vs FNN: TtestResult(statistic=0.5317861822178158, pvalue=0.6230368659919785, df=4)


In [51]:
# Stack all test labels and predictions
y_true = np.concatenate(all_y_test)
y_pred_ensemble = np.concatenate(all_ensemble_pred)
y_prob_ensemble = np.concatenate(all_ensemble_prob)

In [55]:
# --- Classification Report ---
print("\nAverage Classification Report:Ablation static ensemble without WGAN_GP and IMOA")
print(classification_report(y_true, y_pred_ensemble))


Average Classification Report:Ablation static ensemble without WGAN_GP and IMOA
              precision    recall  f1-score   support

           0       0.98      0.88      0.92   6850500
           1       0.37      0.79      0.51    649500

    accuracy                           0.87   7500000
   macro avg       0.68      0.83      0.72   7500000
weighted avg       0.93      0.87      0.89   7500000

